# 第8章：大規模行列の扱い方

バージョン1.10.10で動作確認しています．

事前にインストールが必要なパッケージ
+ LinearAlgebra
+ BenchmarkTools
+ SparseArrays
+ IterativeSolvers
+ Random
+ LinearMaps

## 8.1節：Juliaにおける疎行列の表現

In [ ]:
using LinearAlgebra

In [ ]:
function matvec1(A,x)
    n = length(x)
    y = zeros(n)
    for i = 1:n
        for j = 1:n
            y[i] += A[i,j] * x[j]
        end
    end
    return y
end

function matvec2(A,x)
    n = length(x)
    y = zeros(n)
    for j = 1:n
        for i = 1:n
            y[i] += A[i,j] * x[j]
        end
    end
    return y
end

In [ ]:
n = 10000
A = randn(n,n)
x = ones(n)

In [ ]:
using BenchmarkTools
@belapsed matvec1(A,x)

In [ ]:
@belapsed matvec2(A,x)

In [ ]:
@belapsed A * x

### 疎行列

In [ ]:
A = [1 2 0 0; 0 0 0 1; 0 4 0 1; 0 3 0 1]

In [ ]:
using SparseArrays
A = sparse(A)

In [ ]:
A.m # 行数

In [ ]:
A.n # 列数

In [ ]:
A. colptr # 各列の先頭の非零要素が全体の何番目か

In [ ]:
A.rowval # 各非零要素の行番号

In [ ]:
A.nzval # 各非零要素の値

In [ ]:
function sp_matvec(A,x) 
    n = length(x)
    y = zeros(n)

    @inbounds for j = 1:n
        for k = A.colptr[j]:(A.colptr[j+1]-1)
            i = A.rowval[k]
            y[i] += A.nzval[k] * x[j]
        end
    end

    return y
end


In [ ]:
n = 10000
A = sprandn(n,n,0.02) # 非零要素は約2%
x = randn(n)

In [ ]:
@belapsed sp_matvec(A,x)

In [ ]:
@belapsed A * x

In [ ]:
FullA = Matrix(A) # スパース行列を通常の行列に変換
@belapsed FullA * x

## 8.2.1：定常反復法

In [ ]:
using IterativeSolvers, Random
n = 100
Random.seed!(0) # これを設定すれば，毎回同じ"乱数"が生成される（再現性のため）
B = sprandn(n,n,0.05) # 非零要素は約5%
A = B*B' + 0.01*I # A は正定値行列 (係数行列)

x_exact = ones(n) # 真の解
b = A * x_exact # 右辺ベクトル

In [ ]:
y = sor(A, b, 1.5) # x_exact から程遠い結果になるはず

In [ ]:
y = sor(A, b, 1.5, maxiter=1000) # maxiter を増やすと，x_exact に近づいていくはず

In [ ]:
x = zeros(n) # 初期値
sor!(x, A, b, 1.5, maxiter=1000) # sor! は x を更新するバージョン
x

## 8.2.2節：共役勾配法（CG法）

In [ ]:
using IterativeSolvers, LinearMaps, SparseArrays, Random, BenchmarkTools

n = 1000
Random.seed!(0) # これを設定すれば，毎回同じ"乱数"が生成される（再現性のため）
B = sprandn(n,n,0.05) # 非零要素は約5%
A = B*B' + I # A = B*B' + I は正定値行列 (係数行列)
x_exact = ones(n) # 真の解
b = A*x_exact      # 右辺ベクトル

In [ ]:
A.colptr[end] - 1 # Aの非零要素数

In [ ]:
B.colptr[end] - 1 # Bの非零要素数

In [ ]:
t_min_A_s = @belapsed cg(A, b) # ベンチマークマクロを使って計算時間の最小値を取得

In [ ]:
f!(x) = B*B'*x + x
L = LinearMap(f!,n)

In [ ]:
t_min_L_s = @belapsed cg(L, b) # ベンチマークマクロを使って計算時間の最小値を取得

In [ ]:
t_min_A_s / t_min_L_s # Aを使った場合とLを使った場合の計算時間の比

In [ ]:
# もう少しメモリ効率のよい実装の例

work = zeros(n)  # 再利用ワーク

function g!(y, x)
    mul!(work, adjoint(B), x)   # work = B' * x
    mul!(y,    B,        work)  # y     = B * work
    axpy!(one(eltype(B)), x, y) # y    += x
    y
end

L = LinearMap(g!, n; ismutating=true, issymmetric=true)

In [ ]:
@benchmark cg(L, b)
# result = @benchmark cg(L, b)
# median_s = median(result).time / 1e9